In [ ]:
%env CUDA_VISIBLE_DEVICES=GPU-8868e167-e666-53c7-6c41-d8e83081f07e

In [ ]:
import pandas as pd

#load data
data = pd.read_csv(r"/home/lero/idrive/cmac/DDMAP/Stability studies/Stability_dataset_August_update.csv", na_values='nan')
data = data.drop(data.columns[0:7], axis=1)
data.drop(['Unnamed: 209', 'Unnamed: 210', 'Unnamed: 0'], axis=1, inplace=True)

#data = pd.read_csv(r"/home/lero/idrive/cmac/DDMAP/Stability studies/ML_test_set.csv", na_values='nan')

#Store values for API/ polymer, condition
original_api = data['API']
original_polymer = data['Polymer']
original_condition = data['condition']

#fill pure api with 0 for polymer mol desc
pure = data['Polymer']=='Pure'
polymer_descriptors = data.columns[219:]
data.loc[pure, polymer_descriptors] = 0

#drop conditions since these have been split into temp/ humidity
data.drop(['condition'], axis=1, inplace=True)

#fill na values with average
data.fillna(data.mean(numeric_only=True), inplace=True)

#drop columns with a mean value of 0
numeric_means=data.mean(numeric_only=True)
mean_of_0 = numeric_means[numeric_means==0].index
data.drop(mean_of_0, inplace=True, axis=1)

#Define Features for ColumnTransformer (AFTER ALL DROPS within dataframe) ---
categorical_features = ['API', 'Polymer']
# Identify numerical features:
dont_scale_features = data.drop(['Average days stable', 'GFA'], axis=1).columns.tolist()
numerical_features = [col for col in dont_scale_features if col not in categorical_features]

data.drop(['API', 'Polymer'], inplace=True, axis=1)

data.head()

In [ ]:
#split data
X = data.drop(['Average days stable'], axis=1)
y_non_binary = data['Average days stable']
y = (y_non_binary>=2160).astype(int)

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import SGDClassifier
from catboost import CatBoostClassifier, Pool, cv

models = {
    # 'Logistic Regression': (
    #     LogisticRegression(max_iter=1000000),
    #         {
    #             'model__C': [0.8667904100823],
    #             'model__penalty': ['elasticnet'],
    #             'model__solver': ['saga'],
    #             'model__l1_ratio': [0.9]
    #         }
    # ),
    # 'SVC': (
    #     SVC(max_iter=10000000, random_state=42),
    #     {
    #         'model__C': [1],
    #         'model__kernel': ['poly'],
    #         'model__degree':[1],
    #         'model__gamma': [1],
    #     }
    # ),
    # 'K Neighbors Classifier': (
    #     KNeighborsClassifier(), 
    #     {
    #         'model__n_neighbors': [3]
    #     }
    # ),
    # 'catboost':(
    #     CatBoostClassifier(early_stopping_rounds=100),
    #     {
    #         'model__learning_rate': [0.001, 0.01, 0.05, 0.1],
    #         'model__depth':[3, 10, 50],
    #         'model__iterations': [1000, 1500, 2000],
    #         'model__l2_leaf_reg': [1, 3, 5, 7, 10],
    #         'model__subsample': [0.5, 0.7, 0.9, 1]
    #     }
    # ),
    'Random Forest Classifier': (
        RandomForestClassifier(random_state=42), 
        {
            'model__n_estimators': [300, 500, 1000, 1500, 2000],
            'model__max_features': ['sqrt', 'log2', None],
            'model__max_depth': [None, 10, 50, 100, 300],
            'model__min_samples_split': [2, 5, 10, 20, 50]
        }
    ),
    'XGBoost classifier': (
        XGBClassifier(random_state=42),
        {
            'model__max_depth': [3, 10, 50, 100, 300],
            'model__subsample': [0.5, 0.7, 0.9, 1],
            'model__colsample_bytree': [0.5, 0.7, 0.9, 1],
            'model__learning_rate': [0.001, 0.01, 0.05, 0.1],
            'model__n_estimators': [500, 1000, 1500, 2000],
        }
    ),
    'MLP Classifier': (
        MLPClassifier(max_iter=100000000, early_stopping=True, random_state=42), 
        {
            'model__hidden_layer_sizes': [(100,), (200,), (100, 50), (200, 200)],
            'model__activation': ['logistic', 'tanh', 'relu'],
            'model__alpha': [0.0001, 0.001, 0.01, 0.05, 0.1],
            'model__solver': ['sgd', 'adam'],
            'model__learning_rate': ['constant', 'invscaling', 'adaptive'],
            'model__learning_rate_init': [0.001, 0.01, 0.1],
        }
    )
}

In [ ]:
#pre-processor for one-hot encoding and scaling
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

preprocessor = ColumnTransformer(
    transformers=[ 
        ('num', StandardScaler(), numerical_features),
    ],
    remainder = 'passthrough' # Keep any other columns not explicitly transformed (e.g., if there are any not in num or cat)
)

In [ ]:
#Nested cv
from sklearn.metrics import make_scorer, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, cross_val_score, GroupKFold, cross_val_predict
from sklearn.pipeline import Pipeline
import pickle
from sklearn.metrics import f1_score
import os
from tqdm.auto import tqdm

results = {}
scorer = make_scorer(f1_score, average='binary')

#groups for GroupKFold
groups = (original_api.astype(str)).values

#GroupKFold for outer cv
outer_cv = GroupKFold(n_splits=5) #change n_splits to 80:20
inner_cv = GroupKFold(n_splits=5) #change n_splits to 80:20

#directory to save the models
save_directory = '/home/lero/idrive/cmac/DDMAP/Stability studies/Model_results/Classifier/January_re-run'
os.makedirs(save_directory, exist_ok=True)

for model_name, (classifier, param_grid) in tqdm(models.items(), desc='models', total=len(models)):
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', classifier)
    ])
    
    print('Model:', model_name)
  
    # Perform nested cross-validation
    grid_search = GridSearchCV(estimator=pipeline, param_grid=param_grid, cv=inner_cv, scoring=scorer, verbose=0, n_jobs=100)
    
    fit_params = {'groups': groups}
    
    # Evaluate outer loop scores
    nested_score = cross_val_score(grid_search, X, y, groups=groups, cv=outer_cv, params=fit_params, n_jobs=1)
    
    # Get predictions
    predictions = cross_val_predict(grid_search, X, y, groups=groups, cv=outer_cv, params=fit_params, method='predict', n_jobs=1)

    # Fit to find best parameters
    grid_search.fit(X, y, **fit_params)
    best_params = grid_search.best_params_
    
    # Save the best model
    best_model = grid_search.best_estimator_
    model_file_path = os.path.join(save_directory, f'{model_name}_best_model.pkl')
    with open(model_file_path, 'wb') as model_file:
        pickle.dump(best_model, model_file)

    results[model_name] = {
        'nested_score': nested_score,
        'ground_truth': y.values,
        'predictions': predictions,
        'best_params': best_params,
    }

dictionary_file_path = os.path.join(save_directory, 'Classifiers_results_dictionary.pkl')   
with open(dictionary_file_path, 'wb') as f:
    pickle.dump(results, f)

print('Finito')

In [ ]:
#Ensemble
from sklearn.metrics import make_scorer, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, cross_val_score, GroupKFold, cross_val_predict
from sklearn.pipeline import Pipeline
import pickle
from sklearn.metrics import f1_score
import os
from tqdm.auto import tqdm
from sklearn.ensemble import StackingClassifier
from sklearn.base import clone

##Load pre-trained base models 
with open('/home/lero/idrive/cmac/DDMAP/Stability studies/Model_results/Classifier/January_re-run/Random Forest Classifier_best_model.pkl', 'rb') as f:
    rf_model = pickle.load(f)

with open('/home/lero/idrive/cmac/DDMAP/Stability studies/Model_results/Classifier/January_re-run/XGBoost classifier_best_model.pkl', 'rb') as f:
    XGB_model = pickle.load(f)

with open('/home/lero/idrive/cmac/DDMAP/Stability studies/Model_results/Classifier/January_re-run/MLP Classifier_best_model.pkl', 'rb') as f:
    MLP_model = pickle.load(f)

with open('/home/lero/idrive/cmac/DDMAP/Stability studies/Model_results/Classifier/January_re-run/SVC_best_model.pkl', 'rb') as f:
    SVC_model = pickle.load(f)

with open('/home/lero/idrive/cmac/DDMAP/Stability studies/Model_results/Classifier/January_re-run/Logistic Regression_best_model.pkl', 'rb') as f:
    log_model = pickle.load(f)

groups = (original_api.astype(str)).values

outer_cv = GroupKFold(n_splits=5) 

base_models=[
    ('Random Forest Classifier', rf_model),
    ('XGBoost Classifier', XGB_model),
    ('MLP Classifier', MLP_model),
    ('SVC', SVC_model),
    ('log regression', log_model)
]


#prepare out of fold predictions which have not seen test data
train_oof_predictions = pd.DataFrame(index=X.index)

for name, model in base_models:
    model_clone = clone(model)

    if hasattr(model_clone, 'predict_proba'):
        oof_predictions = cross_val_predict(model_clone, X, y, groups=groups, cv=outer_cv, method='predict_proba', n_jobs=50)[:, 1] #take probabilities only, not n_samples too within 2D array
    else:
        oof_predictions = cross_val_predict(model_clone, X, y, groups=groups, cv=outer_cv, method='decision_function', n_jobs=10)
        
    train_oof_predictions[name]=oof_predictions 


meta_models_tuning = {
    'Logistic Regression': (
        LogisticRegression(max_iter=1000000, random_state=42),
        [
            {  
                'C': np.arange(0.1,2,0.1), 
                'l1_ratio': [0],
                'solver': ['lbfgs', 'newton-cg', 'sag']
            },
            {
                'C': np.arange(0.1,2,0.1),
                'solver': ['liblinear'],
                'l1_ratio': [1],
            },
            {
                'C': np.arange(0.1,2,0.1),
                'penalty': ['elasticnet'],
                'solver': ['saga'],
                'l1_ratio': [0.1, 0.5, 0.9]
            }
        ]
    ),
    'SVC': (
        SVC(max_iter=10000000, random_state=42),
        {
            'C': [0.01, 0.1, 1, 2],
            'kernel': ['poly', 'rbf'],
            'degree':[1,2,3,4,5],
            'gamma': [0.001, 0.01, 0.1, 1],
        }
    ),
    'Random Forest Classifier': (
        RandomForestClassifier(random_state=42), 
        {
            'n_estimators': [10, 50, 100],
            'max_features': ['sqrt', 'log2', None],
            'max_depth': [None, 1, 2, 5, 10, 50],
            'min_samples_split': [2, 5, 10, 20, 50]
        }
    ),
    'XGBoost classifier': (
        XGBClassifier(random_state=42),
        {
            'max_depth': [1, 2, 5, 10, 50],
            'subsample': [0.5, 0.7, 0.9, 1],
            'colsample_bytree': [0.5, 0.7, 0.9, 1],
            'learning_rate': [0.001, 0.01, 0.05],
            'n_estimators': [10, 50, 100],
        }
    ),
}

results = {}
scorer = make_scorer(f1_score, average='binary')

meta_cv = GroupKFold(n_splits=5)
inner_cv = GroupKFold(n_splits=5)

for model_name, (classifier, param_grid) in meta_models_tuning.items():
    print('model:', model_name)
    
    grid_search = GridSearchCV(estimator=classifier, param_grid=param_grid, cv=inner_cv, scoring=scorer, verbose=0, n_jobs=10)

    fit_params = {'groups': groups}
    
    stacked_evaluation_scores = cross_val_score(grid_search, train_oof_predictions, y, groups=groups, params=fit_params, cv=meta_cv, n_jobs=20, scoring=scorer)
    predictions = cross_val_predict(grid_search, train_oof_predictions, y, groups=groups, cv=meta_cv, params=fit_params, method='predict', n_jobs=1)
    
    print(model_name, stacked_evaluation_scores)
    print(f'{model_name} mean stacked scores:', stacked_evaluation_scores.mean())


    #directory to save the models
    save_directory = '/home/lero/idrive/cmac/DDMAP/Stability studies/Model_results/Classifier/January_re-run/stacking'
    os.makedirs(save_directory, exist_ok=True)

    grid_search.fit(train_oof_predictions, y, **fit_params)
    best_params = grid_search.best_params_
    best_model = grid_search.best_estimator_
    

    model_file_path = os.path.join(save_directory, f'{model_name}_best_model.pkl')
    with open(model_file_path, 'wb') as model_file:
        pickle.dump(best_model, model_file)

    results[model_name] = {
        'nested_score': stacked_evaluation_scores,
        'ground_truth': y.values,
        'predictions': predictions,
        'best_params': best_params,
    }

dictionary_file_path = os.path.join(save_directory, 'classifiers_results_dictionary.pkl')   
with open(dictionary_file_path, 'wb') as f:
    pickle.dump(results, f)

print('Finito')

In [ ]:
#model scoring
import pickle
import pandas as pd
import numpy as np
import os

with open('/home/lero/idrive/cmac/DDMAP/Stability studies/Model_results/Classifier/January_re-run/stacking/Classifiers_results_dictionary.pkl', 'rb') as f:
    results = pickle.load(f)
    
records = []

for model in results:
    score = results[model]['nested_score']
    mean_score = np.mean(score)
    records.append({'Model': model, 'Score': mean_score})
    
results_df = pd.DataFrame(records)
results_pivot = results_df.pivot(columns='Model', values='Score')
results_df

In [ ]:
#Visualise model scores
from sklearn.metrics import classification_report
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# List to hold each report entry as a dictionary
reports_list = []

for model in results:
    # Obtain the classification report as a dictionary
    report = classification_report(results[model]['ground_truth'], 
                                   results[model]['predictions'], 
                                   output_dict=True, 
                                   digits=2)
    
    # Flatten the dictionary and add to reports_list
    for class_label, metrics in report.items():
        if isinstance(metrics, dict):  # Ignore the 'accuracy' mean metrics lines
            for metric_name, metric_value in metrics.items():
                reports_list.append({
                    "Model": model,
                    "Class": class_label,
                    "Metric": metric_name,
                    "Value": metric_value
                })

# Convert the list of dictionaries into a DataFrame
reports_df = pd.DataFrame(reports_list)

# Create pivot table to organize data better
pivot_table = reports_df.pivot_table(index=['Model', 'Class'], 
                                     columns='Metric', 
                                     values='Value')

pivot_table.drop(columns='support', inplace=True)

plt.figure(figsize = (8,6), dpi=500)
sns.heatmap(pivot_table, annot = True)
plt.title('Classification Report Metrics Heatmap')
plt.xlabel('Metrics')
plt.xticks(rotation =45)
plt.ylabel('Model and Class')
plt.tight_layout()
#save_directory = '/projects/cp/se_users/ksrn200/Classifier/GroupKFold/Plots/One_week_stability'
#plot_filename = os.path.join(save_directory, 'classification_report.png')
#plt.savefig(plot_filename)

plt.show()


# Display the pivot table
print(pivot_table)

In [ ]:
#visualisation of model performance
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import normalize
import os


for model in results:
    print(model)
    cf_matrix = confusion_matrix(results[model]['ground_truth'], results[model]['predictions'])
    print(cf_matrix)
    print(classification_report(results[model]['ground_truth'], results[model]['predictions'], digits=2))
    plt.figure(figsize=(5,5), dpi=500)
    plt.title(f'{model}: results')
    sns.heatmap(normalize(cf_matrix, axis=1, norm='l1'), annot=True, fmt='.2%', cmap='Blues')
    plt.xlabel('predicted values')
    plt.ylabel('actual values')
    plt.xticks(ticks=[0.5, 1.5], labels=['0', '1'], fontsize=10, rotation=0)
    plt.yticks(ticks=[0.5, 1.5], labels=['0', '1'], fontsize=10, rotation=0)
    
    save_directory = '/home/lero/idrive/cmac/DDMAP/Stability studies/Model_results'
    plot_filename = os.path.join(save_directory, f'{model}_results.png')
    plt.savefig(plot_filename)
    
    plt.show()


In [ ]:
#######--------parameters of importance---------#############
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load the model pipeline
with open('/home/lero/idrive/cmac/DDMAP/Stability studies/Model_results/Classifier/January_re-run/Random Forest Classifier_best_model.pkl', 'rb') as f:
    pipeline = pickle.load(f)
    
# Transform the features
X_transformed = pipeline.named_steps['preprocessor'].transform(X)

model = pipeline.named_steps['model']

importances = model.feature_importances_

feature_names = X.columns

importance_df = pd.DataFrame({'feature': feature_names, 'Importance': importances})
importance_df.sort_values(by = 'Importance', inplace=True, ascending=False)
top_parameters = importance_df.iloc[:20]
print(top_parameters)

plt.figure(figsize=(10,7), dpi=500)
sns.barplot(top_parameters, x='feature', y='Importance')
plt.title('Feature importance using Random Forest classifier')
plt.xticks(rotation =90)
plt.xlabel('Features')
plt.ylabel('Mean accuracy decrease')
plt.tight_layout()
save_directory = '/home/lero/idrive/cmac/DDMAP/Stability studies/Model_results'
plot_filename = os.path.join(save_directory, 'XGBoost_classifier_feature_importance.png')
plt.savefig(plot_filename)
plt.show()


In [ ]:
########---------PCA parameters of importance----------##########
##Inner-CV performance trends, averaged across hyperparameter settings##

import matplotlib.pyplot as plt

save_directory = '/home/lero/idrive/cmac/DDMAP/Stability studies/Model_results/Classifier/January_re-run/PCA'
os.makedirs(save_directory, exist_ok=True)

for model in results:
    print(model)
    
    PCA_summary = results[model]['PCA_n_components_score']
    #print(PCA_summary)
    
    plt.figure(figsize=(4,3), dpi=200)
    plt.errorbar(pca_summary['n_components'], pca_summary['mean'], yerr=pca_summary['std'], capsize=3, marker='o')
    plt.xlabel('n_components')
    plt.ylabel('classification accuracy')
    plt.yticks(np.arange(0,1, step=0.1))
    plt.title(f'{model} performance vs\n PCA dimensionality')
    plt.tight_layout()
    plot_filename = os.path.join(save_directory, f'{model}_performance_vs_pca_n_components.png')
    plt.savefig(plot_filename)
    plt.show